In [1]:
import sys
sys.path.append("..")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_dataset, split_data
from src.models import *
from src.evaluate import *
from src.explain import oof_shap
from src.segment import *
from sklearn.metrics import adjusted_rand_score
from src.explain import *

In [2]:
THRESH = {"telco": 0.36, "bank": 0.32}
KEYCAT = {"telco": ["Contract", "InternetService", "PaymentMethod"], "bank": ["Geography", "Gender"]}

for name in ["telco", "bank"]:
    X_train, X_test, y_train, y_test = split_data(load_dataset(name))
    proba = pd.read_csv(f"../results/oof_xgb_{name}.csv")["proba"].values
    seg = pd.read_csv(f"../results/segments_{name}.csv", index_col=0)["shap_segment"].values
    y = y_train.values
    pred = proba >= THRESH[name]

    df = X_train.copy(); df["segment"] = seg; df["proba"] = proba
    df["outcome"] = np.select([(y == 1) & pred, (y == 1) & ~pred, (y == 0) & pred],
                              ["caught", "missed", "false alarm"], "TN")
    ch = df[y == 1]
    print("=====", name, "=====")
    print("churners:", len(ch), "| missed:", int((ch.outcome == "missed").sum()),
          "| recall:", round((ch.outcome == "caught").mean(), 3))

    print("\nMiss rate by segment:")
    print(ch.groupby("segment").agg(churners=("proba", "size"),
                                    miss_rate=("outcome", lambda s: (s == "missed").mean())).round(3))

    num = list(X_train.select_dtypes("number").columns) + ["proba"]
    print("\nAverage feature values, caught vs missed churners:")
    print(ch.groupby("outcome")[num].mean().round(2).T[["caught", "missed"]])

    for c in KEYCAT[name]:
        print(f"\nShare of churners MISSED, by {c}:")
        print(pd.crosstab(ch[c], ch.outcome, normalize="index")["missed"].round(3))
    df.to_csv(f"../results/error_analysis_{name}.csv")
    print()

===== telco =====
churners: 1495 | missed: 465 | recall: 0.689

Miss rate by segment:
         churners  miss_rate
segment                     
0             229      0.629
1              18      1.000
2             470      0.326
3              24      1.000
4             627      0.011
5             127      0.937

Average feature values, caught vs missed churners:
outcome          caught   missed
SeniorCitizen      0.30     0.15
tenure            11.91    32.63
MonthlyCharges    77.87    68.20
TotalCharges    1070.71  2658.55
proba              0.64     0.19

Share of churners MISSED, by Contract:
Contract
Month-to-month    0.229
One year          0.938
Two year          1.000
Name: missed, dtype: float64

Share of churners MISSED, by InternetService:
InternetService
DSL            0.514
Fiber optic    0.207
No             0.716
Name: missed, dtype: float64

Share of churners MISSED, by PaymentMethod:
PaymentMethod
Bank transfer (automatic)    0.552
Credit card (automatic)      0.55

In [3]:
df = pd.read_csv("../results/error_analysis_telco.csv", index_col=0)
c = df[df.outcome.isin(["caught", "missed"])]
for k in ["One year", "Two year"]:
    s = c[c.Contract == k]
    near = ((s.tenure % 12).isin([0, 1, 11])).mean()
    print(k, "churners:", len(s), "| tenure within 1 month of a multiple of 12:", round(near, 3))
    print(s.tenure.describe().round(1).to_string(), "\n")

One year churners: 130 | tenure within 1 month of a multiple of 12: 0.208
count    130.0
mean      46.1
std       17.7
min        5.0
25%       35.0
50%       49.0
75%       59.0
max       72.0 

Two year churners: 39 | tenure within 1 month of a multiple of 12: 0.333
count    39.0
mean     61.6
std      11.1
min      25.0
25%      56.5
50%      66.0
75%      70.0
max      72.0 

